# 第7章：建立聊天應用程式
## OpenAI API 快速入門

本筆記本改編自 [Azure OpenAI 範例庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)，其中包含存取 [Azure OpenAI](notebook-azure-openai.ipynb) 服務的筆記本。

Python OpenAI API 也可配合 Azure OpenAI 模型使用，只需做少量修改。詳細了解差異請參考：[如何在 Python 中切換 OpenAI 與 Azure OpenAI 端點](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# 概覽  
「大型語言模型是將文字對映到文字的函數。給定一個文字輸入字串，大型語言模型會嘗試預測接下來會出現的文字」(1)。此「快速入門」筆記本將向使用者介紹大型語言模型的高階概念、使用 AML 的核心套件需求、簡單的提示設計入門，以及多個不同使用案例的短範例。 


## 目錄  

[概覽](#overview)  
[如何使用 OpenAI 服務](#how-to-use-openai-service)  
[1. 建立你的 OpenAI 服務](#1.-creating-your-openai-service)  
[2. 安裝](#2.-installation)    
[3. 憑證](#3.-credentials)  

[使用案例](#use-cases)    
[1. 摘要文本](#1.-summarize-text)  
[2. 分類文本](#2.-classify-text)  
[3. 產生新產品名稱](#3.-generate-new-product-names)  
[4. 微調分類器](#4.fine-tune-a-classifier)  

[參考資料](#references)


### 建立你的第一個提示詞  
這個簡短的練習將提供一個基本介紹，教你如何為一個簡單的「摘要」任務提交提示詞給 OpenAI 模型。  


<strong>步驟</strong>：  
1. 在你的 Python 環境中安裝 OpenAI 函式庫  
2. 載入標準輔助函式庫，並為你已建立的 OpenAI 服務設定典型的安全認證  
3. 為你的任務選擇一個模型  
4. 為模型建立一個簡單的提示詞  
5. 將請求提交給模型 API！  


### 1. 安裝 OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. 匯入輔助庫並實例化憑證


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. 尋找合適的模型  
類似 GPT-4o 和 GPT-4o mini 的模型能理解及生成自然語言，並在 OpenAI 平台提供，具有不同程度的運算能力和速度，適合不同的任務。


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. 提示設計  

「大型語言模型的神奇之處在於，透過在大量文本上訓練以最小化此預測錯誤，模型最終學會了對這些預測有用的概念。例如，它們學會了像」(1):

* 如何拼寫
* 語法如何運作
* 如何改寫
* 如何回答問題
* 如何進行對話
* 如何用多種語言寫作
* 如何編程
* 等等。

#### 如何控制大型語言模型  
「在所有大型語言模型的輸入中，最具影響力的莫過於文字提示」(1)。

大型語言模型可以通過幾種方式接收提示來產生輸出：

指令：告訴模型你想要什麼
補全：誘導模型完成你想要的開頭部分
示範：向模型展示你想要什麼，方式包括：
在提示中給出一些例子
在微調訓練數據集中給出數百或數千個例子」



#### 創建提示的三個基本準則：

<strong>展示與說明</strong>。透過指令、範例，或兩者結合，清楚表達你想要什麼。如果你希望模型將一列項目按字母排序或根據情感分類一段文字，就要展示給它看這正是你想要的。

<strong>提供高品質數據</strong>。如果你正嘗試建立分類器或讓模型遵循某種模式，確保有足夠的範例。務必校對範例——模型通常聰明到能識別出基本拼寫錯誤並仍給出回應，但它也可能假設這是故意的，並且會影響回應內容。

**檢查你的設定。** temperature 和 top_p 設定控制模型生成回應的確定性。如果你問的問題只有一個正確答案，那你應該將這些值設低一些。如果你想要更具多樣性的回答，那麼可以設高一些。人們在這些設定上最大的誤解是認為它們是「聰明度」或「創造力」的控制。


來源：https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. 遞交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### 重複執行相同的呼叫，結果有何比較？


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## 總結文本  
#### 挑戰  
通過在文本段落末尾添加「tl;dr:」來總結文本。注意模型如何能在沒有額外指令的情況下理解並執行多項任務。你可以嘗試比 tl;dr 更具描述性的提示詞，以改變模型的行為並定制你所得到的摘要(3)。  

最近的研究展示了通過先在大量文本語料上預訓練，然後在特定任務上微調，能在許多 NLP 任務和基準測試中取得顯著提升。雖然該方法通常在架構上是任務不可知的，但仍需要含有數千甚至數萬個樣本的任務特定微調數據集。相比之下，人類通常只需少量示例或簡單指令即可執行新的語言任務 — 這是現有 NLP 系統仍然很難做到的。在此我們展示，擴大語言模型規模能大幅提升任務不可知的少樣本性能，有時甚至能達到以往最先進微調方法的競爭水平。  



總結  


# 多個使用情境練習  
1. 摘要文本  
2. 文字分類  
3. 產生新產品名稱


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 分類文本  
#### 挑戰  
將項目分類到推理時間提供的類別中。在以下範例中，我們在提示中同時提供了類別和要分類的文本(*playground_reference)。 

客戶查詢：您好，我的筆記本鍵盤的一個按鍵最近壞了，我需要一個替換的按鍵：

分類類別：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 產生新產品名稱
#### 挑戰
從範例詞語創建產品名稱。這裡我們在提示中包含了我們將為之命名的產品資訊。我們還提供了類似的範例以展示我們希望獲得的模式。我們還將 temperature 值設置得較高，以增加隨機性和更具創意的回應。

產品描述：家用奶昔機
種子詞：快速，健康，緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：一雙可適合任何腳尺寸的鞋子。
種子詞：適應性，合腳，全尺碼適應。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# 參考資料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [微調 GPT 模型以分類文本的最佳實務](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 如需更多協助  
[OpenAI 商業化團隊](AzureOpenAITeam@microsoft.com) 


# 貢獻者
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
